# Lecture 14 — Implementing the NDArray Backend

**CMU 10-414/714, Deep Learning Systems (lecture by Tianqi Chen)**
Companion notebook · Sangram Lembe

Until homework 3, needle's arrays were NumPy arrays. This lecture replaces them with needle's own
**NDArray**: one flat buffer of memory, plus a few numbers describing how to read it as a
multi-dimensional array. This notebook builds a small but complete version in Python:

1. the five fields, and the path a call like `x + 1` travels
2. views — reshape, slice, transpose, broadcast — that copy nothing
3. the bug that happens when a kernel is handed a view, and `compact()`, which prevents it
4. adding a new operator, including the lecture's copy-paste bug, caught by a test

The real backends are C++ and CUDA; the "kernels" here are Python loops over a flat buffer, which is
exactly the view those backends have.

In [1]:
import numpy as np
from itertools import product

class Backend:
    """Stands in for the C++ / CUDA module. Kernels only ever see FLAT buffers."""
    def __init__(self, name):
        self.name, self.log = name, []

    def array(self, size):                       # allocate a flat buffer
        return np.empty(size, dtype=np.float32)

    def from_numpy(self, src, out):              # host -> device copy (cudaMemcpy on a GPU)
        out[:] = src.ravel()

    def scalar_add(self, a, val, out):
        self.log.append("scalar_add kernel")
        for i in range(out.size):                # CPU: a loop.  CUDA: one thread per i
            out[i] = a[i] + val

    def ewise_add(self, a, b, out):
        self.log.append("ewise_add kernel")
        for i in range(out.size):
            out[i] = a[i] + b[i]

    def ewise_mul(self, a, b, out):
        for i in range(out.size):
            out[i] = a[i] * b[i]

cpu = Backend("cpu")
print("backend ready")

backend ready


## 1. The NDArray: a flat handle plus four numbers

`shape`, `strides` and `offset` describe how to read the flat `handle`; `device` says which backend
owns it. The element at index $(i_0, i_1, \ldots)$ lives at
$\text{offset} + \sum_k i_k \cdot \text{stride}_k$.

In [2]:
def compact_strides(shape):
    """Row-major strides for a tightly packed array: (3, 1) for shape (2, 3)."""
    strides, s = [], 1
    for d in reversed(shape):
        strides.append(s); s *= d
    return tuple(reversed(strides))

class NDArray:
    copies = 0                                   # count real data copies, for the demos

    def __init__(self, data, device=cpu):
        data = np.asarray(data, dtype=np.float32)
        self._init(data.shape, compact_strides(data.shape), 0, device, None)
        device.from_numpy(data, self._handle)

    def _init(self, shape, strides, offset, device, handle):
        self.shape, self.strides, self.offset, self.device = tuple(shape), tuple(strides), offset, device
        self._handle = handle if handle is not None else device.array(int(np.prod(shape)))

    @staticmethod
    def make(shape, strides=None, device=cpu, handle=None, offset=0):
        """Build an NDArray from parts. With a handle given, NOTHING is copied."""
        a = NDArray.__new__(NDArray)
        a._init(shape, strides or compact_strides(shape), offset, device, handle)
        return a

    def numpy(self):
        """Read every element through the strides (device -> host copy on a GPU)."""
        out = np.empty(self.shape, dtype=np.float32)
        for idx in product(*[range(d) for d in self.shape]):
            out[idx] = self._handle[self.offset + sum(i * s for i, s in zip(idx, self.strides))]
        return out

    def __repr__(self):
        return f"NDArray({self.numpy().tolist()}, shape={self.shape}, strides={self.strides}, offset={self.offset})"

x = NDArray([[0, 1, 2], [3, 4, 5]])
print(x)
print("flat handle:", x._handle)

NDArray([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0]], shape=(2, 3), strides=(3, 1), offset=0)
flat handle: [0. 1. 2. 3. 4. 5.]


## 2. Views: new shape and strides, same memory

Each of these returns a new NDArray pointing at the **same** handle.

In [3]:
def reshape(self, new_shape):
    """Only valid on packed memory - otherwise the flat order is not the logical order."""
    packed = self.offset == 0 and self.strides == compact_strides(self.shape)
    if not packed:
        raise ValueError("reshape needs a compact array: call compact() first")
    return NDArray.make(new_shape, device=self.device, handle=self._handle)

def permute(self, axes):                         # transpose = permute((1, 0))
    return NDArray.make([self.shape[a] for a in axes], [self.strides[a] for a in axes],
                        self.device, self._handle, self.offset)

def broadcast_to(self, new_shape):               # new dimensions get stride 0
    pad = len(new_shape) - len(self.shape)
    strides = [0] * pad + [0 if (d == 1 and n != 1) else s
               for d, n, s in zip(self.shape, new_shape[pad:], self.strides)]
    return NDArray.make(new_shape, strides, self.device, self._handle, self.offset)

def getitem(self, slices):                       # slices: one (start, stop, step) per dimension
    offset = self.offset + sum(st * s for (st, _, _), s in zip(slices, self.strides))
    shape = [len(range(st, sp, sp_)) for st, sp, sp_ in slices]
    strides = [s * sp_ for (_, _, sp_), s in zip(slices, self.strides)]
    return NDArray.make(shape, strides, self.device, self._handle, offset)

NDArray.reshape, NDArray.permute = reshape, permute
NDArray.broadcast_to, NDArray.getitem = broadcast_to, getitem

In [4]:
flat = NDArray(np.arange(6))
A = flat.reshape((2, 3))
views = {
    "A (reshape of 6)":         (A,                                   np.arange(6).reshape(2, 3)),
    "A[1:2, :] (second row)":   (A.getitem([(1, 2, 1), (0, 3, 1)]),   np.arange(6).reshape(2, 3)[1:2, :]),
    "A[:, 1:] (slice)":         (A.getitem([(0, 2, 1), (1, 3, 1)]),   np.arange(6).reshape(2, 3)[:, 1:]),
    "A.T (permute)":            (A.permute((1, 0)),                   np.arange(6).reshape(2, 3).T),
}
for name, (v, ref) in views.items():
    print(f"{name:<24} shape {str(v.shape):<8} strides {str(v.strides):<8} offset {v.offset}"
          f"   same memory: {v._handle is flat._handle}   correct: {np.array_equal(v.numpy(), ref)}")

A (reshape of 6)         shape (2, 3)   strides (3, 1)   offset 0   same memory: True   correct: True
A[1:2, :] (second row)   shape (1, 3)   strides (3, 1)   offset 3   same memory: True   correct: True
A[:, 1:] (slice)         shape (2, 2)   strides (3, 1)   offset 1   same memory: True   correct: True
A.T (permute)            shape (3, 2)   strides (1, 3)   offset 0   same memory: True   correct: True


Broadcasting is the strangest view: stride 0 means moving along that dimension costs nothing, so the
array can be **bigger than the memory behind it**.

In [5]:
row = NDArray([7, 8, 9])
big = row.reshape((1, 3)).broadcast_to((4, 3))
print("memory behind it:", row._handle.size, "numbers")
print("array it pretends to be:", big.shape, "=", int(np.prod(big.shape)), "numbers,  strides", big.strides)
print(big.numpy())

memory behind it: 3 numbers
array it pretends to be: (4, 3) = 12 numbers,  strides (0, 1)
[[7. 8. 9.]
 [7. 8. 9.]
 [7. 8. 9.]
 [7. 8. 9.]]


## 3. The bug `compact()` prevents

A kernel only receives the flat handle and walks it from the start. Hand it a view, and it reads the
wrong numbers. Take `A[:, 1:]`, which holds 1, 2, 4, 5:

In [6]:
S = A.getitem([(0, 2, 1), (1, 3, 1)])            # the view A[:, 1:]
print("the view holds      :", S.numpy().ravel().tolist())
print("correct answer of +1:", (S.numpy() + 1).ravel().tolist())

out = NDArray.make(S.shape)
cpu.scalar_add(S._handle, 1.0, out._handle)      # WRONG: raw handle straight to the kernel
print("kernel on raw handle:", out.numpy().ravel().tolist(), "  <- read slots 0-3 of the buffer")

the view holds      : [1.0, 2.0, 4.0, 5.0]
correct answer of +1: [2.0, 3.0, 5.0, 6.0]
kernel on raw handle: [1.0, 2.0, 3.0, 4.0]   <- read slots 0-3 of the buffer


`compact()` fixes it. If the array is already packed in row-major order with offset 0, it does
nothing. Otherwise it copies the view into a fresh, tightly packed buffer — this is the function
homework 3 asks you to write for arbitrary shapes.

In [7]:
def is_compact(self):
    return self.offset == 0 and self.strides == compact_strides(self.shape) \
           and self._handle.size == int(np.prod(self.shape))

def compact(self):
    if is_compact(self):
        return self                               # lazy: nothing to do
    NDArray.copies += 1
    out = NDArray.make(self.shape, device=self.device)
    for k, idx in enumerate(product(*[range(d) for d in self.shape])):
        out._handle[k] = self._handle[self.offset + sum(i * s for i, s in zip(idx, self.strides))]
    return out

NDArray.is_compact, NDArray.compact = is_compact, compact

def __add__(self, other):
    """y = x + other: pick the kernel, allocate the output, compact the inputs."""
    out = NDArray.make(self.shape, device=self.device)
    if isinstance(other, NDArray):
        self.device.ewise_add(self.compact()._handle, other.compact()._handle, out._handle)
    else:
        self.device.scalar_add(self.compact()._handle, float(other), out._handle)
    return out
NDArray.__add__ = __add__

print("S + 1 with compact():", (S + 1).numpy().ravel().tolist())

S + 1 with compact(): [2.0, 3.0, 5.0, 6.0]


### Compaction is lazy

Views chain for free. Only the operation that finally needs real numbers pays for one copy.

In [8]:
NDArray.copies = 0
base = NDArray(np.arange(12)).reshape((3, 4))
v = base.getitem([(0, 3, 1), (1, 4, 1)])        # slice      -> free
v = v.permute((1, 0))                            # transpose  -> free
v = NDArray.make((2,) + v.shape, (0,) + v.strides, v.device, v._handle, v.offset)   # broadcast -> free
print("copies after slice + transpose + broadcast:", NDArray.copies)
result = v + 1
print("copies after one addition:                 ", NDArray.copies)
print("matches NumPy:", np.array_equal(result.numpy(),
      np.broadcast_to(np.arange(12).reshape(3, 4)[:, 1:].T, (2, 3, 3)) + 1))

copies after slice + transpose + broadcast: 0
copies after one addition:                  1
matches NumPy: True


### Tracing a call

With logging on, `x + 1` shows the path from Python operator to kernel.

In [9]:
cpu.log.clear()
x = NDArray([1, 2, 3])
y = x + 1
print("y =", y.numpy().tolist())
print("path: Python __add__ -> scalar or array? -> make(output) -> x.compact() ->", cpu.log)

y = [2.0, 3.0, 4.0]
path: Python __add__ -> scalar or array? -> make(output) -> x.compact() -> ['scalar_add kernel']


## 4. Adding an operator — and the lecture's bug

Tianqi adds element-wise division live, starting from a copy of the multiplication kernel. The first
version forgot to change the operation. A test against NumPy catches it at once.

In [10]:
def ewise_div_buggy(a, b, out):
    for i in range(out.size):
        out[i] = a[i] * b[i]                     # copied from ewise_mul... and not changed

def ewise_div(a, b, out):
    for i in range(out.size):
        out[i] = a[i] / b[i]

def test_div(kernel):
    a_np, b_np = np.random.rand(5).astype(np.float32) + 1, np.random.rand(5).astype(np.float32) + 1
    a, b = NDArray(a_np), NDArray(b_np)
    out = NDArray.make(a.shape)
    kernel(a.compact()._handle, b.compact()._handle, out._handle)
    return np.allclose(out.numpy(), a_np / b_np)

print("buggy division passes test:", test_div(ewise_div_buggy))
print("fixed division passes test:", test_div(ewise_div))

buggy division passes test: False
fixed division passes test: True


The lecture's second lesson here: when you rebuild a compiled backend, a running notebook still has
the **old** library loaded. Run tests in a fresh Python process (`python test.py`) so they see the new
code.

## Summary

- An NDArray is a flat `handle` plus `shape`, `strides`, `offset` and `device`; the device picks the
  NumPy, C++ or CUDA backend.
- Reshape, slice, transpose and broadcast only change those numbers — every view above shared memory
  with the original, and a broadcast array can be larger than its memory.
- Kernels walk the flat buffer, so a view must be compacted first: `A[:,1:] + 1` gave 1, 2, 3, 4 from
  the raw handle instead of 2, 3, 5, 6.
- `compact()` is lazy: three chained views cost zero copies, and the addition cost one.
- Test every new kernel against a reference — the copy-pasted division kernel failed immediately.